# 2026.10.19(월) 20h45~

In [1]:
import os                         # 환경변수 읽기용
from dotenv import load_dotenv    # .env 파일을 읽어 환경변수로 등록하는 함수

load_dotenv()  # .env 파일에서 OPENAI_API_KEY를 읽어 옴

# 키가 없으면 여기서 바로 멈추고 안내 메시지 출력 (뒤에서 알기 어려운 에러가 나는 것을 방지)
assert os.getenv("OPENAI_API_KEY"), ".env 파일에 OPENAI_API_KEY가 없습니다."

# 실습 전체에서 사용할 모델 (gpt-4o-mini는 2027-01-14 종료 예정, 이후에는 여기만 바꾸면 됨)
MODEL = "gpt-4o-mini"

## (강사님 원본 파일에 이 섹션 제목이 뭐였더라? >.<)

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model=MODEL, temperature=0)

# 짧은 질문용: 질문을 그대로 전달
short_prompt = ChatPromptTemplate.from_template("{question}")
# 긴 질문용: 검색 결과를 요약하듯 답하도록 지시 (실제 검색은 하지 않음)
search_prompt = ChatPromptTemplate.from_template("인터넷 검색을 통해 '{question}'에 대해 요약해 주세요.")

# 테스트 질문: 첫 번째는 20자 이하, 두 번째는 20자 초과
questions = [
    "한국의 수도는 어디인가요?",
    "대한민국의 정치 체계와 역사적 변천 과정을 설명해 주세요.",
]

In [3]:
from langchain_core.runnables import RunnableLambda   # 일반 함수를 체인 부품으로 바꿔 주는 클래스

short_chain = short_prompt | llm    # 짧은 질문용 체인
long_chain = search_prompt | llm    # 긴 질문용 체인

# 입력을 보고 어떤 체인을 실행할지 골라서 돌려주는 함수
def route(input_dict):
    if len(input_dict["question"]) > 20:    # 질문 글자 수로 판단
        print("[긴 질문으로 판단]")
        return long_chain
    print("[짧은 질문으로 판단]")
    return short_chain

final_chain = RunnableLambda(route)   # 입력을 보고 실행할 체인을 돌려줌 (돌려받은 체인이 이어서 실행됨)

for q in questions:
    output = final_chain.invoke({"question": q})
    print("질문:", q)
    print("답변:", output.content, "\n")

[짧은 질문으로 판단]
질문: 한국의 수도는 어디인가요?
답변: 한국의 수도는 서울입니다. 

[긴 질문으로 판단]
질문: 대한민국의 정치 체계와 역사적 변천 과정을 설명해 주세요.
답변: 대한민국의 정치 체계와 역사적 변천 과정은 다음과 같이 요약할 수 있습니다.

1. **일제 강점기 (1910-1945)**: 일본의 식민지 지배 아래에서 한국은 정치적 자율성을 상실했습니다. 이 시기에 민족주의 운동과 독립운동이 활발히 전개되었습니다.

2. **광복과 분단 (1945-1948)**: 제2차 세계대전 종전 후, 한국은 미국과 소련에 의해 남북으로 분단되었습니다. 1948년, 남한에서는 대한민국 정부가 수립되었습니다.

3. **초기 민주 정부 (1948-1960)**: 이승만 대통령 하에 민주주의 체제가 시작되었으나, 권위주의적 통치와 부정선거로 인해 사회적 불만이 커졌습니다. 1960년 4.19 혁명으로 이승만 정부가 무너졌습니다.

4. **군사 정권 (1961-1987)**: 박정희가 1961년 군사 쿠데타로 권력을 잡고 경제 개발을 추진했습니다. 그러나 인권 탄압과 정치적 억압이 심화되었습니다. 1980년 광주 민주화 운동이 발생했습니다.

5. **민주화 과정 (1987-현재)**: 1987년 6월 민주항쟁을 통해 직선제 개헌이 이루어졌고, 이후 민주주의가 정착되었습니다. 여러 차례의 대통령 선거와 정치적 변화가 있었으며, 현재는 다당제와 자유로운 선거가 이루어지고 있습니다.

6. **현대 정치 체계**: 대한민국은 대통령제를 채택하고 있으며, 국회는 단원제로 구성되어 있습니다. 정치적 다양성과 시민 참여가 강조되고 있으며, 민주주의와 인권이 중요한 가치로 자리잡고 있습니다.

이러한 과정은 대한민국의 정치 체계가 역사적 사건과 사회적 요구에 따라 변화해왔음을 보여줍니다. 



# 2026.10.19(월) 21h~

## LangGraph 조건부 edge

lambda statement(표현식) = 임시 함수, 함수는 아닌데 함수 역할을 함

In [4]:
from typing import Optional
from pydantic import BaseModel
from langgraph.graph import StateGraph, START, END

# State: 질문, 답변, 그리고 분기 판단 결과까지 함께 저장
class QAState(BaseModel):
    question: str
    answer: Optional[str] = None
    need_search: Optional[bool] = None   # 분기 판단 결과

# 노드 1: 질문 길이로 검색 필요 여부 판단 (답변은 만들지 않고 판단 결과만 기록)
def classify_question(state: QAState):
    print("classify_question 실행")
    return {"need_search": len(state.question) > 20}

# 노드 2: 짧은 질문 → 바로 답변
def direct_answer(state: QAState):
    print("direct_answer 실행")
    return {"answer": llm.invoke(short_prompt.format_messages(question=state.question)).content}

# 노드 3: 긴 질문 → 검색 요약 스타일로 답변
def search_answer(state: QAState):
    print("search_answer 실행")
    return {"answer": llm.invoke(search_prompt.format_messages(question=state.question)).content}

graph = StateGraph(QAState)
graph.add_node("classify", classify_question)
graph.add_node("direct_answer", direct_answer)
graph.add_node("search_answer", search_answer)

graph.add_edge(START, "classify")   # 항상 분류부터 시작

# 조건부 엣지: classify가 끝나면 State를 보고 다음 노드를 결정
graph.add_conditional_edges(
    "classify",                                                              # 출발 노드
    lambda state: "search_answer" if state.need_search else "direct_answer", # 다음 노드 이름을 돌려주는 함수
    ["search_answer", "direct_answer"],   # 갈 수 있는 노드 목록
)
graph.add_edge("direct_answer", END)   # 어느 쪽이든 답변 후 종료
graph.add_edge("search_answer", END)

app = graph.compile()
print(app.get_graph().draw_mermaid())   # 그래프 구조를 Mermaid 텍스트로 확인

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	direct_answer(direct_answer)
	search_answer(search_answer)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	classify -.-> direct_answer;
	classify -.-> search_answer;
	direct_answer --> __end__;
	search_answer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



![조건 분기 그래프](langgraph_classify_graph.png)

In [5]:
results = []   # 확인 셀에서 쓰기 위해 실행 결과를 모아 둠
for q in questions:
    result = app.invoke({"question": q})   # 질문만 넣으면 나머지는 그래프가 채움
    results.append(result)
    print("질문:", q)
    print("답변:", result["answer"], "\n")

classify_question 실행
direct_answer 실행
질문: 한국의 수도는 어디인가요?
답변: 한국의 수도는 서울입니다. 

classify_question 실행
search_answer 실행
질문: 대한민국의 정치 체계와 역사적 변천 과정을 설명해 주세요.
답변: 대한민국의 정치 체계와 역사적 변천 과정은 다음과 같이 요약할 수 있습니다.

1. **일제 강점기 (1910-1945)**: 일본의 식민지 지배 아래에서 한국은 정치적 자율성을 상실했습니다. 이 시기에 민족주의 운동과 독립운동이 활발히 전개되었습니다.

2. **광복과 분단 (1945-1948)**: 제2차 세계대전 종전 후, 한국은 미국과 소련에 의해 남북으로 분단되었습니다. 1948년, 남한에서는 대한민국 정부가 수립되었습니다.

3. **초기 민주 정부 (1948-1960)**: 이승만 대통령 하에 민주주의 체제가 시작되었으나, 권위주의적 통치와 부정선거로 인해 사회적 불만이 커졌습니다. 1960년 4.19 혁명으로 이승만 정부가 무너졌습니다.

4. **군사 정권 (1961-1987)**: 박정희가 1961년 군사 쿠데타로 권력을 잡고 경제 개발을 추진했습니다. 그러나 인권 탄압과 정치적 억압이 심화되었습니다. 1980년 광주 민주화 운동이 발생했습니다.

5. **민주화 과정 (1987-현재)**: 1987년 6월 민주항쟁을 통해 직선제 개헌이 이루어졌고, 이후 민주주의가 정착되었습니다. 여러 차례의 대통령 선거와 정치적 변화가 있었으며, 최근에는 정치적 다양성과 시민 참여가 증가하고 있습니다.

6. **현재의 정치 체계**: 대한민국은 대통령제를 채택하고 있으며, 국회는 단원제로 구성되어 있습니다. 정치적 정당이 다양하게 존재하고, 민주적 절차를 통해 정부가 운영되고 있습니다.

이러한 과정은 대한민국의 정치적 정체성과 민주주의 발전에 중요한 영향을 미쳤습니다. 



cf. 강사님 원본 파일에는 '정리' 같은 섹션 있는데, 안 보여주셨다 ㅠㅠ